# QTDB single-noise checkpoints → zero-shot RMN benchmark

Notebook đánh giá các checkpoint DDPM-BW, DDPM-EM và DDPM-MA hiện tại trên test set Random Mixed Noise chính thức của `fully_gated_DAE`.

Kết quả phải được gọi là **zero-shot/transfer RMN**, không phải so sánh cùng điều kiện huấn luyện với FGDAE. Notebook cũng tạo artifact-specific subset benchmark trên các hàng RMN thuần BW, EM hoặc MA.

### Kaggle Inputs

- Hai checkpoint `best_<artifact>_protocol1/2_norm_loss_v2.pth` cho artifact đang chạy.
- `Dataset_1.pkl` và `Dataset_2.pkl` nếu đã có; nếu thiếu, notebook sẽ tạo lại từ QTDB/NSTDB và cần bật Internet.
- Tùy chọn: `joint_metrics_per_segment.csv` để so sánh trên đúng segment với FGDAE và các baseline.

Mặc định chạy một artifact để tránh hết thời gian. Lần lượt đặt `('BW',)`, `('EM',)` và `('MA',)`. Sau ba phiên, chạy một phiên tổng hợp với `RUN_INFERENCE=False`, `REQUIRE_ALL_ZERO_SHOT_MODELS=True` và thêm ba output làm Input.

## 1. Cài thư viện

In [ ]:
!pip -q install 'wfdb>=4.1,<5' 'PyWavelets>=1.5' 'prettytable>=3.9'


## 2. Cấu hình

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import pickle
import platform
import random
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 1234
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Run one artifact per Kaggle version to stay within the execution limit.
MODEL_ARTIFACTS_TO_EVALUATE = ('BW',)
NOISE_VERSIONS = (1, 2)  # verified mapping: P1=nv1, P2=nv2
RUN_INFERENCE = True
RUN_COMPARISON = True
REQUIRE_ALL_ZERO_SHOT_MODELS = False

# None means the official full 15,535-segment test set. Use 128 only for smoke tests.
MAX_TEST_SEGMENTS = None
INFERENCE_BATCH_SIZE = 50
DDIM_STEPS = 50
DDIM_ETA = 0.0
K = 1
SAVE_PREDICTIONS = False

if RUN_INFERENCE and DEVICE.type != 'cuda':
    raise RuntimeError('Enable GPU in Kaggle before zero-shot diffusion inference.')

INPUT_ROOT = Path('/kaggle/input')
TEMP_ROOT = Path('/kaggle/temp/qtdb_zero_shot_rmn')
DATA_DIR = TEMP_ROOT / 'data' / '4_RMN'
REPO_DIR = TEMP_ROOT / 'fully_gated_DAE'
OUT = Path('/kaggle/working/qtdb_zero_shot_rmn')
METRIC_DIR = OUT / 'metrics'
REPORT_DIR = OUT / 'reports'
PREDICTION_DIR = OUT / 'predictions'
for directory in (DATA_DIR, METRIC_DIR, REPORT_DIR, PREDICTION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TARGET_FS = 360
LENGTH = 512
PREPROCESSING_VERSION = 'qtdb_pu1_p_and_beat_40ms_reflect100ms_360hz_endpoints_pad16_v2'
MODEL_NORMALIZATION = 'per_segment_clean_maxabs_shared_with_noisy_v1'
LOSS_VERSION = 'l1_mean_relative_stft_snr_weighted_continuous_alpha_v1'
CHECKPOINT_TAG = 'norm_loss_v2'
CONFIG = {
    'train': {'feats': 80, 'batch_size': 96, 'epochs': 100, 'lr': 1e-4,
              'lambda_time': 1.0, 'lambda_freq': 0.1},
    'diffusion': {'beta_start': 1e-4, 'beta_end': 0.5,
                  'num_steps': 50, 'schedule': 'quad'},
    'model_normalization': MODEL_NORMALIZATION,
    'loss_version': LOSS_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
}

FGDAE_REPO_URL = 'https://github.com/AhmedAShaheen/fully_gated_DAE.git'
FGDAE_REPO_COMMIT = '70e768e'
VALID_ARTIFACTS = ('BW', 'EM', 'MA')
if not set(MODEL_ARTIFACTS_TO_EVALUATE).issubset(VALID_ARTIFACTS):
    raise ValueError(MODEL_ARTIFACTS_TO_EVALUATE)
if set(NOISE_VERSIONS) - {1, 2}:
    raise ValueError(NOISE_VERSIONS)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('Device:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU (aggregation only)')
print('Model artifacts:', MODEL_ARTIFACTS_TO_EVALUATE, '| RMN versions:', NOISE_VERSIONS)


## 3. Chuẩn bị RMN Dataset 1/2

In [ ]:
def attached_dataset_directory():
    parents = sorted({path.parent for path in INPUT_ROOT.rglob('Dataset_1.pkl')
                      if (path.parent / 'Dataset_2.pkl').is_file()})
    if not parents:
        return None
    # Identical duplicate pairs are still ambiguous at the path level; keep one input only.
    if len(parents) != 1:
        raise RuntimeError('Multiple Dataset_1/2 directories found; keep one: ' + str(parents))
    return parents[0]


def clone_repository():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', FGDAE_REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', 'checkout', FGDAE_REPO_COMMIT], cwd=REPO_DIR, check=True)
    prep = REPO_DIR / 'Data_Preparation' / 'data_preparation_overlapped.py'
    lines = prep.read_text(encoding='utf-8').splitlines()
    patched = []
    for line in lines:
        if 'test_name = re.search' in line:
            indent = line[:len(line) - len(line.lstrip())]
            line = indent + "test_name = str(signal_name).replace('\\\\', '/').split('/')[-1]"
        patched.append(line)
    prep.write_text('\n'.join(patched) + '\n', encoding='utf-8')


def download_zip(url, destination, attempts=5):
    if destination.is_file() and zipfile.is_zipfile(destination):
        return
    partial = destination.with_suffix(destination.suffix + '.part')
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 ECG-benchmark/1.0'})
            with urllib.request.urlopen(request, timeout=180) as response, open(partial, 'wb') as output:
                shutil.copyfileobj(response, output, length=1024 * 1024)
            with zipfile.ZipFile(partial) as archive:
                if archive.testzip() is not None:
                    raise zipfile.BadZipFile('Archive integrity test failed')
            partial.replace(destination)
            return
        except Exception as error:
            if partial.exists():
                partial.unlink()
            if attempt == attempts:
                raise RuntimeError(
                    f'Cannot download {url}. Attach Dataset_1.pkl and Dataset_2.pkl as Kaggle Input.'
                ) from error
            time.sleep(min(30, 3 * 2 ** (attempt - 1)))


def ensure_datasets():
    source = attached_dataset_directory()
    if source is not None:
        for version in (1, 2):
            shutil.copy2(source / f'Dataset_{version}.pkl', DATA_DIR / f'Dataset_{version}.pkl')
        print('Reused RMN datasets from:', source)
        return
    print('No attached RMN datasets; generating the official QTDB RMN datasets.')
    clone_repository()
    repo_data = REPO_DIR / 'data'
    repo_output = repo_data / '4_RMN'
    repo_output.mkdir(parents=True, exist_ok=True)
    downloads = {
        'qt-database-1.0.0.zip': 'https://physionet.org/static/published-projects/qtdb/qt-database-1.0.0.zip',
        'mit-bih-noise-stress-test-database-1.0.0.zip':
            'https://physionet.org/static/published-projects/nstdb/mit-bih-noise-stress-test-database-1.0.0.zip',
    }
    for name, url in downloads.items():
        archive_path = REPO_DIR / name
        download_zip(url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(repo_data)
    subprocess.run([sys.executable, 'generate_data.py', '--Data', 'RMN1'],
                   cwd=REPO_DIR, check=True)
    for version in (1, 2):
        shutil.copy2(repo_output / f'Dataset_{version}.pkl', DATA_DIR / f'Dataset_{version}.pkl')


if RUN_INFERENCE:
    ensure_datasets()
    DATASET_PATHS = {version: DATA_DIR / f'Dataset_{version}.pkl' for version in (1, 2)}
else:
    DATASET_PATHS = {}
    print('Aggregation-only mode: RMN datasets are not required.')
for version, path in DATASET_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f'nv{version}: {path.stat().st_size / 1024**2:.1f} MiB')


## 4. Audit dataset

In [ ]:
def load_dataset(version):
    with open(DATASET_PATHS[version], 'rb') as handle:
        payload = pickle.load(handle)
    if len(payload) != 7:
        raise ValueError(f'nv{version}: expected 7 items, got {len(payload)}')
    return payload


dataset_rows = []
for version in DATASET_PATHS:
    x_train, y_train, x_test, y_test, noise_scale, noise_mask, subject_names = load_dataset(version)
    if x_train.shape != y_train.shape or x_test.shape != y_test.shape:
        raise ValueError(f'nv{version}: noisy/clean mismatch')
    if x_test.shape[1:] != (512, 1):
        raise ValueError(f'nv{version}: expected (N,512,1), got {x_test.shape}')
    if not (len(x_test) == len(noise_scale) == len(noise_mask) == len(subject_names)):
        raise ValueError(f'nv{version}: metadata length mismatch')
    dataset_rows.append({
        'noise_version': f'nv{version}', 'train_segments': len(x_train),
        'test_segments': len(x_test), 'shape': str(x_test.shape),
        'dtype': str(x_test.dtype),
        'clean_test_sha256': hashlib.sha256(np.asarray(y_test).tobytes()).hexdigest(),
        'noisy_test_sha256': hashlib.sha256(np.asarray(x_test).tobytes()).hexdigest(),
    })
    del x_train, y_train, x_test, y_test, noise_scale, noise_mask, subject_names
    gc.collect()
dataset_audit = pd.DataFrame(dataset_rows)
dataset_audit.to_csv(REPORT_DIR / 'dataset_audit.csv', index=False)
display(dataset_audit)


## 5. Kiến trúc checkpoint hiện tại

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
import torch
import torch.nn as nn

class HNFBlock(nn.Module):
    """
    Half Normalized Filter block.
    Multi-scale convs → concat → conv → half‑instance norm → residual.
    """
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k,
                      padding=k//2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        # Half‑instance norm: chia channel làm 2 nửa
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        multi_out = [conv(x) for conv in self.multi_convs]
        out = torch.cat(multi_out, dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out_norm = self.half_inst_norm(out[:, :half, :])
        out = torch.cat([out_norm, out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)
import torch
import torch.nn as nn
import math

class BridgeBlock(nn.Module):
    """FiLM‑based conditioning on noise level (sqrt(alpha_bar))."""
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(
            nn.Linear(emb_dim, features * 2),
            nn.SiLU()
        )

    def forward(self, x, alpha_bar):
        # alpha_bar: (B,) or scalar
        # Ép phẳng alpha_bar về dạng 1D (Batch,) để tránh bị dư chiều -- CỰC QUAN TRỌNG
        alpha_bar = alpha_bar.view(-1)
        emb = self.sinusoidal_embedding(alpha_bar)
        scale_shift = self.film(emb)   # (B, 2*features)
        scale, shift = scale_shift.chunk(2, dim=1)
        scale = scale.unsqueeze(-1)
        shift = shift.unsqueeze(-1)
        return x * (1 + scale) + shift

    def sinusoidal_embedding(self, x):
        if x.dim() == 0:
            x = x.unsqueeze(0)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb
import torch
import torch.nn as nn

class SelfAttention1D(nn.Module):
    """1D Self‑Attention (multi‑head) dùng tại bottleneck."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels, "channels phải chia hết cho num_heads"

        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, L = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, L)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        # q, k, v: (B, num_heads, head_dim, L)
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        out = out.reshape(B, C, L)
        return self.proj(out)
import torch
import torch.nn as nn

class UNet1D(nn.Module):
    """
    1D U‑Net với 4 levels, skip connections, HNF blocks,
    Bridge FiLM (timestep), và Self‑Attention tại bottleneck.
    """
    def __init__(self, in_channels=24, base_channels=64, emb_dim=128, out_channels=12):
        """
        in_channels = 24 vì concat x_t (12 channels) và condition (12 channels)
        """
        super().__init__()
        # ---------- ENCODER ----------
        # Level 1 (input -> 64)
        self.enc1 = HNFBlock(in_channels, base_channels)
        self.bridge1 = BridgeBlock(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels*2, kernel_size=4, stride=2, padding=1)

        # Level 2 (64 -> 128)
        self.enc2 = HNFBlock(base_channels*2, base_channels*2)
        self.bridge2 = BridgeBlock(base_channels*2, emb_dim)
        self.down2 = nn.Conv1d(base_channels*2, base_channels*4, kernel_size=4, stride=2, padding=1)

        # Level 3 (128 -> 256)
        self.enc3 = HNFBlock(base_channels*4, base_channels*4)
        self.bridge3 = BridgeBlock(base_channels*4, emb_dim)
        self.down3 = nn.Conv1d(base_channels*4, base_channels*8, kernel_size=4, stride=2, padding=1)

        # Level 4 (256 -> 512) – bottleneck
        self.enc4 = HNFBlock(base_channels*8, base_channels*8)
        self.bridge4 = BridgeBlock(base_channels*8, emb_dim)
        # Self‑Attention tại bottleneck
        self.attn = SelfAttention1D(base_channels*8)

        # ---------- DECODER ----------
        # Level 4 -> 3
        self.up4 = nn.ConvTranspose1d(base_channels*8, base_channels*4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlock(base_channels*8, base_channels*4)   # concat với skip từ enc3

        # Level 3 -> 2
        self.up3 = nn.ConvTranspose1d(base_channels*4, base_channels*2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlock(base_channels*4, base_channels*2)   # concat với skip từ enc2

        # Level 2 -> 1
        self.up2 = nn.ConvTranspose1d(base_channels*2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlock(base_channels*2, base_channels)     # concat với skip từ enc1

        # Output
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        """
        x: latent (x_t), cond: noisy observation, noise_scale: sqrt(alpha_bar)
        """
        # Concatenate x and cond along channel dimension
        inp = torch.cat([x, cond], dim=1)   # (B, 2, L)

        # ----- Encoder -----
        e1 = self.enc1(inp)
        e1 = self.bridge1(e1, noise_scale)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        e2 = self.bridge2(e2, noise_scale)
        d2 = self.down2(e2)

        e3 = self.enc3(d2)
        e3 = self.bridge3(e3, noise_scale)
        d3 = self.down3(e3)

        e4 = self.enc4(d3)
        e4 = self.bridge4(e4, noise_scale)
        # Bottleneck self‑attention
        e4 = self.attn(e4)

        # ----- Decoder (with skip connections) -----
        u4 = self.up4(e4)
        u4 = torch.cat([u4, e3], dim=1)   # skip từ enc3
        d4 = self.dec4(u4)

        u3 = self.up3(d4)
        u3 = torch.cat([u3, e2], dim=1)   # skip từ enc2
        d3 = self.dec3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, e1], dim=1)   # skip từ enc1
        d2 = self.dec2(u2)

        out = self.final(d2)
        return out

## 6. Conditional DDPM/DDIM

In [ ]:
from functools import partial
from inspect import isfunction
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from inspect import isfunction

# ----- Bắt buộc phải có torchaudio hoặc dùng torch.stft -----
# Nếu không có torchaudio, ta dùng torch.stft nguyên bản
def stft_loss(pred, target, n_fft=128, hop_length=64):
    """
    Tính MSE loss trên phổ STFT giữa pred và target.
    """
    # (B, C, L) -> (B*C, L)
    B, C, L = pred.shape
    pred = pred.view(-1, L)
    target = target.view(-1, L)
    
    # STFT: returns (B, freq_bins, time_frames, 2)
    spec_pred = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, 
                           return_complex=True)
    spec_target = torch.stft(target, n_fft=n_fft, hop_length=hop_length,
                             return_complex=True)
    
    # Lấy magnitude (độ lớn)
    mag_pred = torch.abs(spec_pred)
    mag_target = torch.abs(spec_target)
    
    return F.mse_loss(mag_pred, mag_target)


def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


class DDPM(nn.Module):
    def __init__(self, base_model, config, device, conditional=True):
        super().__init__()
        self.device = device
        self.model = base_model
        self.config = config
        self.conditional = conditional
        
        # ----------------------
        # 👉 THÊM: Loss weights
        # ----------------------
        self.lambda_time = config['train'].get('lambda_time', 1.0)
        self.lambda_freq = config['train'].get('lambda_freq', 0.1)  # cân bằng
        
        self.loss_func_l1 = nn.L1Loss(reduction='sum').to(device)
        
        config_diff = config["diffusion"]
        self.num_steps = config_diff["num_steps"]
        self.set_new_noise_schedule(config_diff, device)
        
    def make_beta_schedule(self, schedule='linear', n_timesteps=1000, start=1e-5, end=1e-2):
        if schedule == 'linear':
            betas = torch.linspace(start, end, n_timesteps)
        elif schedule == "quad":
            betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
        elif schedule == "sigmoid":
            betas = torch.linspace(-6, 6, n_timesteps)
            betas = torch.sigmoid(betas) * (end - start) + start
        return betas
    
    def set_new_noise_schedule(self, config_diff, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)
        
        betas = self.make_beta_schedule(
            schedule=config_diff["schedule"], 
            n_timesteps=config_diff["num_steps"],
            start=config_diff["beta_start"], 
            end=config_diff["beta_end"]
        )
        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas
        
        alphas = 1. - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod_prev = np.sqrt(np.append(1., alphas_cumprod))
        
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev', to_torch(alphas_cumprod_prev))
        self.register_buffer('sqrt_alphas_cumprod', to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('log_one_minus_alphas_cumprod', to_torch(np.log(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod - 1)))
        
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        self.register_buffer('posterior_variance', to_torch(posterior_variance))
        self.register_buffer('posterior_log_variance_clipped', 
                             to_torch(np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', 
                             to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', 
                             to_torch((1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))
        
    def predict_start_from_noise(self, x_t, t, noise):
        # Nắn lại chiều tensor hệ số thành (Batch, 1, 1) để nhân đúng với tensor tín hiệu 3D
        coef1 = self.sqrt_recip_alphas_cumprod[t].view(-1, 1, 1)
        coef2 = self.sqrt_recipm1_alphas_cumprod[t].view(-1, 1, 1)
        
        return coef1 * x_t - coef2 * noise
    
    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * x_start + \
                         self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]
        return posterior_mean, posterior_log_variance_clipped
    
    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):
        batch_size = x.shape[0]
        noise_level = torch.FloatTensor(
            [self.sqrt_alphas_cumprod_prev[t+1]]
        ).repeat(batch_size, 1).to(x.device)
        if condition_x is not None:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, condition_x, noise_level))
        else:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, noise_level))
        if clip_denoised:
            x_recon.clamp_(-1., 1.)
        model_mean, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance
    
    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised=False, condition_x=None):
        model_mean, model_log_variance = self.p_mean_variance(
            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()

    @torch.no_grad()
    def p_sample_loop(self, x_in, continous=False):
        device = self.betas.device
        sample_inter = (1 | (self.num_steps//10))
        if not self.conditional:
            shape = x_in
            cur_x = torch.randn(shape, device=device)
            ret_x = cur_x
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i)
                if i % sample_inter == 0:
                    ret_x = torch.cat([ret_x, cur_x], dim=0)
        else:
            x = x_in
            shape = x.shape
            cur_x = torch.randn(shape, device=device)
            ret_x = [cur_x]
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i, condition_x=x)
                if i % sample_inter == 0:
                    ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]    
    
    @torch.no_grad()
    def sample(self, batch_size=1, shape=[1, 512], continous=False):
        return self.p_sample_loop((batch_size, shape[0], shape[1]), continous)
    
    # ==========================================================
    # 👉 DDIM SAMPLER (đã có sẵn trong source gốc)
    # ==========================================================
    @torch.no_grad()
    def ddim_sample_loop(self, x_in, ddim_timesteps=50, ddim_eta=0.0, num_shots=1):
        device = self.betas.device
        T = self.num_steps
        S = ddim_timesteps
        tau = [int(np.floor((T / (S ** 2)) * (i ** 2))) for i in range(S + 1)]
        
        alphas_cumprod_with_1 = torch.cat([
            torch.tensor([1.0], dtype=self.alphas_cumprod.dtype, device=device),
            self.alphas_cumprod
        ]).float()
        
        out_accum = 0.0
        for shot in range(num_shots):
            shape = x_in.shape
            x = torch.randn(shape, device=device)
            for i in reversed(range(1, S + 1)):
                t = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.FloatTensor(
                    [self.sqrt_alphas_cumprod_prev[t]]
                ).repeat(x.shape[0], 1).to(device)
                if not self.conditional:
                    eps = self.model(x, noise_level)
                else:
                    eps = self.model(x, x_in, noise_level)
                    
                a_t = alphas_cumprod_with_1[t]
                a_t_prev = alphas_cumprod_with_1[t_prev]
                x0_pred = (x - torch.sqrt(1. - a_t) * eps) / torch.sqrt(a_t)
                if t_prev == 0:
                    sigma_t = 0.0
                else:
                    sigma_t = ddim_eta * torch.sqrt((1. - a_t_prev) / (1. - a_t)) * \
                              torch.sqrt(torch.clamp(1. - a_t / a_t_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1. - a_t_prev - sigma_t**2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_t_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots

    @torch.no_grad()
    def denoising(self, x_in, continous=False, use_ddim=False, ddim_steps=50, ddim_eta=0.0, num_shots=1):
        if use_ddim:
            return self.ddim_sample_loop(x_in, ddim_timesteps=ddim_steps, ddim_eta=ddim_eta, num_shots=num_shots)
        else:
            if num_shots > 1:
                out_accum = 0.0
                for _ in range(num_shots):
                    out_accum += self.p_sample_loop(x_in, continous)
                return out_accum / num_shots
            return self.p_sample_loop(x_in, continous)
    
    def q_sample_loop(self, x_start, continous=False):
        sample_inter = (1 | (self.num_steps//10))
        ret_x = [x_start]
        cur_x = x_start
        for t in range(1, self.num_steps+1):
            B,C,L = cur_x.shape
            continuous_sqrt_alpha_cumprod = torch.FloatTensor(
                np.random.uniform(
                    self.sqrt_alphas_cumprod_prev[t-1],
                    self.sqrt_alphas_cumprod_prev[t],
                    size=B
                )
            ).to(cur_x.device)
            continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
            noise = torch.randn_like(cur_x)
            cur_x = self.q_sample(
                x_start=cur_x, 
                continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
                noise=noise
            )
            if t % sample_inter == 0:
                ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]
    
    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):
        noise = default(noise, lambda: torch.randn_like(x_start))
        return continuous_sqrt_alpha_cumprod * x_start + \
               (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise
    
    # ==========================================================
    # 👉 HÀM LOSS CHÍNH (ĐÃ THÊM STFT)
    # ==========================================================
    def p_losses(self, x_in, y_in, noise=None):
        """
        Multi‑domain Loss:
        - L1 trên noise (miền thời gian)
        - STFT Loss trên tín hiệu dự đoán (miền tần số)
        """
        x_start = x_in
        B, C, L = x_start.shape

        # Lấy ngẫu nhiên các giá trị t khác nhau cho từng mẫu trong Batch, trực tiếp trên thiết bị (GPU)
        t = torch.randint(0, self.num_steps, (B,), device=x_start.device).long()
        # 1. Chuyển mảng NumPy sang PyTorch Tensor và đẩy lên cùng GPU với t
        sqrt_alphas_prev_tensor = torch.tensor(self.sqrt_alphas_cumprod_prev, dtype=torch.float32, device=t.device)
        # 2. Lấy giới hạn dưới và trên. 
        # (Lưu ý: Do t giờ chạy từ 0 đến num_steps-1, ta dùng chỉ số t và t+1 để thay thế cho t-1 và t nhằm tránh lỗi âm)
        lower = sqrt_alphas_prev_tensor[t]
        upper = sqrt_alphas_prev_tensor[t+1]
        # 3. Sinh số ngẫu nhiên uniform hoàn toàn bằng PyTorch trực tiếp trên GPU
        continuous_sqrt_alpha_cumprod = lower + torch.rand(B, device=t.device) * (upper - lower)
        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
        noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(
            x_start=x_start, 
            continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
            noise=noise
        )

        # Dự đoán noise
        if not self.conditional:
            x_recon = self.model(x_noisy, continuous_sqrt_alpha_cumprod)
        else:
            x_recon = self.model(x_noisy, y_in, continuous_sqrt_alpha_cumprod)

        # ---------- 1. L1 Loss (miền thời gian) ----------
        loss_l1 = self.loss_func_l1(noise, x_recon)  # (B*C*L)

        # ---------- 2. STFT Loss (miền tần số) ----------
        # Dự đoán x0 từ noise đã dự đoán
        x0_pred = self.predict_start_from_noise(x_noisy, t, x_recon)
        
        # Tính STFT loss giữa x0_pred và x_start (clean gốc)
        loss_stft = stft_loss(x0_pred, x_start, n_fft=128, hop_length=64)
        
        # ---------- 3. Tổng hợp ----------
        total_loss = self.lambda_time * loss_l1 + self.lambda_freq * loss_stft
        return total_loss
    
    def forward(self, x, y, *args, **kwargs):
        return self.p_losses(x, y, *args, **kwargs)


class EMA(object):
    def __init__(self, mu=0.999):
        self.mu = mu
        self.shadow = {}

    def register(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name].data = (1. - self.mu) * param.data + self.mu * self.shadow[name].data

    def ema(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name].data)

    def ema_copy(self, module):
        module_copy = type(module)(module.config).to(module.config.device)
        module_copy.load_state_dict(module.state_dict())
        self.ema(module_copy)
        return module_copy

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state_dict):
        self.shadow = state_dict

class ResearchLossDDPM(DDPM):
    """Same DDPM/UNet architecture, with a scale-stable multi-domain objective."""

    def loss_components(self, x_in, y_in, noise=None):
        x_start = x_in
        batch = x_start.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=x_start.device)
        boundaries = torch.as_tensor(
            self.sqrt_alphas_cumprod_prev, dtype=x_start.dtype, device=x_start.device
        )
        lower, upper = boundaries[timestep], boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=x_start.device) * (upper - lower)
        continuous_2d = continuous.view(batch, 1)
        continuous_3d = continuous.view(batch, 1, 1)
        gaussian_noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(x_start, continuous_3d, gaussian_noise)
        predicted_noise = self.model(x_noisy, y_in, continuous_2d)

        # Mean keeps the time loss independent of batch size and 512-sample length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Reconstruct x0 with the exact continuous alpha used to create x_t.
        residual_scale = torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0))
        x0_prediction = (x_noisy - residual_scale * predicted_noise) / continuous_3d.clamp_min(1e-6)

        n_fft, hop_length = 128, 64
        window = torch.hann_window(n_fft, dtype=x_start.dtype, device=x_start.device)
        pred_flat = x0_prediction.reshape(-1, x0_prediction.shape[-1])
        target_flat = x_start.reshape(-1, x_start.shape[-1])
        pred_mag = torch.abs(torch.stft(pred_flat, n_fft=n_fft, hop_length=hop_length,
                                         window=window, return_complex=True))
        target_mag = torch.abs(torch.stft(target_flat, n_fft=n_fft, hop_length=hop_length,
                                           window=window, return_complex=True))
        pred_mag = pred_mag.reshape(batch, -1)
        target_mag = target_mag.reshape(batch, -1)
        spectral_error = torch.mean((pred_mag - target_mag) ** 2, dim=1)
        target_power = torch.mean(target_mag ** 2, dim=1).clamp_min(1e-6)
        relative_frequency = spectral_error / target_power

        # x0 is ill-conditioned at high-noise timesteps, so weight STFT by diffusion SNR.
        alpha_bar = continuous ** 2
        diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
        frequency_weight = torch.clamp(diffusion_snr, max=1.0)
        loss_frequency = torch.mean(relative_frequency * frequency_weight)
        total = self.lambda_time * loss_time + self.lambda_freq * loss_frequency
        return {'total': total, 'time': loss_time, 'frequency': loss_frequency,
                'frequency_unweighted': relative_frequency.mean(),
                'frequency_weight': frequency_weight.mean()}

    def p_losses(self, x_in, y_in, noise=None):
        return self.loss_components(x_in, y_in, noise)['total']


## 7. Kiểm tra checkpoint và mapping P1/nv1, P2/nv2

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def resolve_checkpoint(artifact, protocol):
    filename = f'best_{artifact.lower()}_protocol{protocol}_{CHECKPOINT_TAG}.pth'
    matches = sorted(INPUT_ROOT.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f'Need exactly one {filename}; found {matches}')
    return matches[0]


checkpoint_rows = []
if RUN_INFERENCE:
    for artifact in MODEL_ARTIFACTS_TO_EVALUATE:
        for version in NOISE_VERSIONS:
            path = resolve_checkpoint(artifact, version)
            payload = torch.load(path, map_location='cpu', weights_only=False)
            if payload.get('artifact') != artifact or int(payload.get('protocol', -1)) != version:
                raise RuntimeError(f'Checkpoint identity mismatch: {path}')
            if payload.get('config') != CONFIG:
                raise RuntimeError(f'Checkpoint configuration mismatch: {path}')
            if 'ema_model_state_dict' not in payload:
                raise RuntimeError(f'EMA inference weights missing: {path}')
            checkpoint_rows.append({
                'train_artifact': artifact, 'protocol': version, 'noise_version': f'nv{version}',
                'selected_epoch': int(payload['epoch']) + 1,
                'best_validation_loss': float(payload['best_validation_loss']),
                'sha256': sha256(path), 'path': str(path),
            })
else:
    print('Aggregation-only mode: checkpoints are not required.')
checkpoint_manifest = pd.DataFrame(checkpoint_rows)
checkpoint_manifest.to_csv(REPORT_DIR / 'checkpoint_manifest.csv', index=False)
display(checkpoint_manifest)


## 8. Metric chung

In [ ]:
def artifact_labels(mask):
    names = np.array(['BW', 'MA', 'EM'])
    labels = []
    for row in np.asarray(mask).astype(bool):
        active = names[row]
        labels.append('Clean' if len(active) == 0 else '+'.join(active))
    return labels


def calculate_metrics(clean, noisy, prediction, eps=1e-12):
    clean, noisy, prediction = [np.asarray(value, dtype=np.float64).reshape(len(value), -1)
                                for value in (clean, noisy, prediction)]
    error = prediction - clean
    input_error = noisy - clean
    squared = error ** 2
    ssd = squared.sum(axis=1)
    signal_power = (clean ** 2).sum(axis=1)
    input_power = (input_error ** 2).sum(axis=1)
    centered = clean - clean.mean(axis=1, keepdims=True)
    standard_denominator = (centered ** 2).sum(axis=1)
    repo_denominator = ((prediction - clean.mean()) ** 2).sum(axis=1)
    norm_product = np.linalg.norm(clean, axis=1) * np.linalg.norm(prediction, axis=1)
    snr_in = 10 * np.log10(np.maximum(signal_power, eps) / np.maximum(input_power, eps))
    snr_out = 10 * np.log10(np.maximum(signal_power, eps) / np.maximum(ssd, eps))
    return pd.DataFrame({
        'SSD': ssd,
        'MAD_mV': np.abs(error).max(axis=1),
        'PRD_paper_pct': 100 * np.sqrt(ssd / np.maximum(standard_denominator, eps)),
        'PRD_repo_pct': 100 * np.sqrt(ssd / np.maximum(repo_denominator, eps)),
        'CosSim': (clean * prediction).sum(axis=1) / np.maximum(norm_product, eps),
        'RMSE_mV': np.sqrt(squared.mean(axis=1)),
        'MAE_mV': np.abs(error).mean(axis=1),
        'SNR_in_dB': snr_in,
        'SNR_dB': snr_out,
        'SNR_improvement_dB': snr_out - snr_in,
    })


## 9. Zero-shot RMN inference

In [ ]:
def build_model():
    return ResearchLossDDPM(UNet1D(2, 80, 128, 1), CONFIG, DEVICE).to(DEVICE)


def test_count(array):
    return len(array) if MAX_TEST_SEGMENTS is None else min(len(array), int(MAX_TEST_SEGMENTS))


run_rows = []
if RUN_INFERENCE:
    for artifact in MODEL_ARTIFACTS_TO_EVALUATE:
        output_frames = []
        for version in NOISE_VERSIONS:
            checkpoint_path = resolve_checkpoint(artifact, version)
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
            model = build_model()
            model.load_state_dict(checkpoint['ema_model_state_dict'], strict=True)
            model.eval()

            payload = load_dataset(version)
            x_test, y_test, noise_scale, noise_mask, subject_names = payload[2:]
            count = test_count(x_test)
            noisy = np.asarray(x_test[:count], dtype=np.float32)[:, :, 0]
            clean = np.asarray(y_test[:count], dtype=np.float32)[:, :, 0]
            scale = np.maximum(np.max(np.abs(clean), axis=1, keepdims=True), 1e-8).astype(np.float32)

            predictions = np.lib.format.open_memmap(
                PREDICTION_DIR / f'xhat_model_{artifact.lower()}_rmn_nv{version}_K{K}.npy',
                mode='w+', dtype=np.float32, shape=clean.shape,
            ) if SAVE_PREDICTIONS else np.empty_like(clean)

            artifact_seed = {'BW': 11, 'EM': 22, 'MA': 33}[artifact]
            torch.manual_seed(SEED + artifact_seed * 100 + version)
            torch.cuda.manual_seed_all(SEED + artifact_seed * 100 + version)
            started = time.perf_counter()
            for start in tqdm(range(0, count, INFERENCE_BATCH_SIZE),
                              desc=f'DDPM-{artifact} zero-shot RMN nv{version}'):
                stop = min(start + INFERENCE_BATCH_SIZE, count)
                condition = torch.from_numpy((noisy[start:stop] / scale[start:stop])[:, None, :]).to(DEVICE)
                with torch.inference_mode():
                    output = model.denoising(condition, use_ddim=True, ddim_steps=DDIM_STEPS,
                                             ddim_eta=DDIM_ETA, num_shots=K)
                if output.shape != condition.shape or not torch.isfinite(output).all():
                    raise RuntimeError(f'Invalid output {artifact}/nv{version}/{start}')
                predictions[start:stop] = output.cpu().numpy()[:, 0, :] * scale[start:stop]
            elapsed = time.perf_counter() - started
            if hasattr(predictions, 'flush'):
                predictions.flush()

            metric_frame = calculate_metrics(clean, noisy, predictions)
            metric_frame.insert(0, 'checkpoint_sha256', sha256(checkpoint_path))
            metric_frame.insert(0, 'segment_index', np.arange(count))
            metric_frame.insert(0, 'subject', list(subject_names)[:count])
            labels = artifact_labels(np.asarray(noise_mask)[:count])
            metric_frame.insert(0, 'artifact', labels)
            metric_frame.insert(0, 'test_artifact', labels)
            metric_frame.insert(0, 'noise_scale', np.asarray(noise_scale)[:count])
            metric_frame.insert(0, 'protocol', version)
            metric_frame.insert(0, 'noise_version', f'nv{version}')
            metric_frame.insert(0, 'train_artifact', artifact)
            metric_frame.insert(0, 'model', f'DDPM-{artifact} zero-shot RMN')
            output_frames.append(metric_frame)
            run_rows.append({
                'model': f'DDPM-{artifact} zero-shot RMN', 'train_artifact': artifact,
                'noise_version': f'nv{version}', 'protocol': version, 'segments': count,
                'inference_seconds': elapsed, 'segments_per_second': count / elapsed,
                'checkpoint': str(checkpoint_path), 'checkpoint_sha256': sha256(checkpoint_path),
                'full_test': MAX_TEST_SEGMENTS is None,
            })
            del model, checkpoint, payload, x_test, y_test, noisy, clean, scale, predictions
            torch.cuda.empty_cache()
            gc.collect()
        artifact_output = pd.concat(output_frames, ignore_index=True)
        artifact_output.to_csv(METRIC_DIR / f'zero_shot_metrics_ddpm_{artifact.lower()}.csv', index=False)
        print(artifact, 'metric rows:', len(artifact_output))
    pd.DataFrame(run_rows).to_csv(REPORT_DIR / 'inference_runs.csv', index=False)
else:
    print('Inference disabled; attached metric CSVs will be aggregated.')


## 10. Ghép các phiên BW/EM/MA

In [ ]:
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


metric_candidates = list(INPUT_ROOT.rglob('zero_shot_metrics_ddpm_*.csv'))
metric_candidates.extend(METRIC_DIR.glob('zero_shot_metrics_ddpm_*.csv'))
selected_files = {}
for path in metric_candidates:
    name = path.name
    digest = file_hash(path)
    if name in selected_files and selected_files[name][0] != digest:
        raise RuntimeError(f'Conflicting versions of {name}: {selected_files[name][1]} and {path}')
    selected_files[name] = (digest, path)

if not selected_files:
    raise FileNotFoundError('No zero_shot_metrics_ddpm_*.csv files found')
zero_shot = pd.concat([pd.read_csv(item[1]) for item in selected_files.values()], ignore_index=True)
identity = ['model', 'noise_version', 'segment_index']
if zero_shot.duplicated(identity).any():
    raise RuntimeError('Duplicate zero-shot model/version/segment rows')
available_artifacts = set(zero_shot.train_artifact.unique())
if REQUIRE_ALL_ZERO_SHOT_MODELS and available_artifacts != set(VALID_ARTIFACTS):
    raise RuntimeError(f'Need BW, EM, MA metrics; found {sorted(available_artifacts)}')
zero_shot.to_csv(REPORT_DIR / 'zero_shot_metrics_all.csv', index=False)
print('Available zero-shot models:', sorted(available_artifacts), '| rows:', len(zero_shot))

COMMON_METRICS = ['SSD', 'MAD_mV', 'PRD_paper_pct', 'PRD_repo_pct',
                  'CosSim', 'RMSE_mV', 'MAE_mV', 'SNR_dB']


def summary_table(frame, group_columns):
    means = frame.groupby(group_columns)[COMMON_METRICS].mean().add_suffix('_mean')
    stds = frame.groupby(group_columns)[COMMON_METRICS].std(ddof=0).add_suffix('_std')
    counts = frame.groupby(group_columns).size().rename('N')
    return pd.concat([counts, means, stds], axis=1).reset_index()


full_summary = summary_table(zero_shot, ['model', 'train_artifact'])
matching = zero_shot[zero_shot.test_artifact == zero_shot.train_artifact].copy()
subset_summary = summary_table(matching, ['model', 'train_artifact', 'test_artifact'])
composition_summary = summary_table(zero_shot, ['model', 'train_artifact', 'test_artifact'])
full_summary.to_csv(REPORT_DIR / 'zero_shot_summary_full_rmn.csv', index=False)
subset_summary.to_csv(REPORT_DIR / 'zero_shot_summary_matching_subset.csv', index=False)
composition_summary.to_csv(REPORT_DIR / 'zero_shot_summary_by_composition.csv', index=False)
display(full_summary)
display(subset_summary)


## 11. Artifact-specific subset benchmark với baseline

In [ ]:
baseline_paths = sorted(INPUT_ROOT.rglob('joint_metrics_per_segment.csv'))
if RUN_COMPARISON and baseline_paths:
    digests = {file_hash(path): path for path in baseline_paths}
    if len(digests) != 1:
        raise RuntimeError(f'Conflicting joint_metrics_per_segment.csv files: {baseline_paths}')
    baseline = pd.read_csv(next(iter(digests.values())))
    baseline = baseline[baseline.model != 'MyDDPM'].copy()
    required = {'model', 'noise_version', 'segment_index', 'artifact', *COMMON_METRICS}
    missing = required - set(baseline.columns)
    if missing:
        raise ValueError(f'Baseline evidence missing columns: {sorted(missing)}')

    expected_keys = zero_shot[['noise_version', 'segment_index']].drop_duplicates()
    for model_name, model_frame in baseline.groupby('model'):
        observed = model_frame[['noise_version', 'segment_index']].drop_duplicates()
        if len(observed.merge(expected_keys)) != len(expected_keys):
            raise RuntimeError(f'Baseline {model_name} does not cover every zero-shot test row')

    common_zero = zero_shot[['model', 'noise_version', 'segment_index', 'artifact', *COMMON_METRICS]].copy()
    full_comparison = pd.concat([baseline[['model', 'noise_version', 'segment_index', 'artifact', *COMMON_METRICS]],
                                 common_zero], ignore_index=True)
    full_comparison.to_csv(REPORT_DIR / 'comparison_metrics_per_segment.csv', index=False)
    comparison_summary = summary_table(full_comparison, ['model'])
    comparison_summary.to_csv(REPORT_DIR / 'comparison_summary_full_rmn.csv', index=False)

    subset_frames = []
    for artifact in sorted(available_artifacts):
        zero_part = common_zero[(zero_shot.train_artifact == artifact).to_numpy() &
                                (common_zero.artifact == artifact)]
        baseline_part = baseline[baseline.artifact == artifact][
            ['model', 'noise_version', 'segment_index', 'artifact', *COMMON_METRICS]
        ].copy()
        zero_part = zero_part.copy()
        baseline_part['evaluation_subset'] = artifact
        zero_part['evaluation_subset'] = artifact
        subset_frames.extend([baseline_part, zero_part])
    subset_comparison = pd.concat(subset_frames, ignore_index=True)
    subset_comparison.to_csv(REPORT_DIR / 'comparison_matching_subset_per_segment.csv', index=False)
    subset_comparison_summary = summary_table(subset_comparison, ['evaluation_subset', 'model'])
    subset_comparison_summary.to_csv(REPORT_DIR / 'comparison_matching_subset_summary.csv', index=False)
    display(subset_comparison_summary)
else:
    baseline = None
    print('Baseline comparison skipped. Add joint_metrics_per_segment.csv as Kaggle Input to enable it.')


## 12. Biểu đồ

In [ ]:
COLORS = {'BW': '#4C78A8', 'EM': '#F58518', 'MA': '#54A24B'}
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': 0.2, 'font.size': 10,
                     'axes.titleweight': 'bold'})

fig, axes = plt.subplots(1, 2, figsize=(14, max(4.5, 0.55 * len(full_summary))))
for axis, metric, title, direction in (
        (axes[0], 'PRD_paper_pct_mean', 'Full RMN: standard PRD (%)', 'lower'),
        (axes[1], 'SNR_dB_mean', 'Full RMN: output SNR (dB)', 'higher')):
    ordered = full_summary.sort_values(metric, ascending=(direction == 'lower'))
    positions = np.arange(len(ordered))
    axis.barh(positions, ordered[metric], color=[COLORS[a] for a in ordered.train_artifact])
    axis.set(yticks=positions, yticklabels=ordered.model, xlabel=f'{title} — {direction} is better')
    axis.invert_yaxis()
fig.suptitle('QTDB single-artifact DDPM checkpoints — zero-shot RMN', y=1.02)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'zero_shot_full_rmn_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

compositions = ['Clean', 'BW', 'MA', 'BW+MA', 'EM', 'BW+EM', 'MA+EM', 'BW+MA+EM']
matrix = composition_summary.pivot(index='model', columns='test_artifact',
                                   values='SNR_dB_mean').reindex(columns=compositions)
fig, axis = plt.subplots(figsize=(13, max(3.5, 0.8 * len(matrix))))
image = axis.imshow(matrix.to_numpy(), aspect='auto', cmap='YlGnBu')
axis.grid(False)
axis.set(xticks=np.arange(len(matrix.columns)), xticklabels=matrix.columns,
         yticks=np.arange(len(matrix.index)), yticklabels=matrix.index,
         xlabel='RMN artifact composition', ylabel='Zero-shot model',
         title='Mean output SNR by RMN artifact composition')
axis.tick_params(axis='x', rotation=30)
for row in range(len(matrix.index)):
    for column in range(len(matrix.columns)):
        value = matrix.iloc[row, column]
        if np.isfinite(value):
            axis.text(column, row, f'{value:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=axis, label='Mean SNR-out (dB)')
fig.tight_layout()
fig.savefig(REPORT_DIR / 'zero_shot_snr_by_rmn_composition.png', dpi=180, bbox_inches='tight')
plt.show()

if baseline is not None:
    focus_models = ['FGDAE q=1', 'FGDAE q=2', 'FGDAE q=3', 'FGDAE q=4'] + sorted(zero_shot.model.unique())
    plot_data = subset_comparison_summary[subset_comparison_summary.model.isin(focus_models)].copy()
    artifacts = sorted(plot_data.evaluation_subset.unique())
    fig, axes = plt.subplots(1, len(artifacts), figsize=(6 * len(artifacts), 5), squeeze=False)
    for axis, artifact in zip(axes.ravel(), artifacts):
        part = plot_data[plot_data.evaluation_subset == artifact].sort_values('PRD_paper_pct_mean')
        positions = np.arange(len(part))
        colors = [COLORS[artifact] if name.startswith('DDPM-') else '#7A8793' for name in part.model]
        axis.barh(positions, part.PRD_paper_pct_mean, color=colors)
        axis.set(yticks=positions, yticklabels=part.model,
                 xlabel='Mean standard PRD (%) — lower is better', title=f'Test subset: {artifact}')
        axis.invert_yaxis()
    fig.suptitle('Artifact-specific subset: zero-shot DDPM vs FGDAE', y=1.02)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'zero_shot_vs_fgdae_matching_subsets.png', dpi=180, bbox_inches='tight')
    plt.show()


## 13. Đóng gói

In [ ]:
run_info = {
    'model_artifacts_requested': list(MODEL_ARTIFACTS_TO_EVALUATE),
    'model_artifacts_available': sorted(available_artifacts),
    'noise_versions': list(NOISE_VERSIONS),
    'protocol_mapping': {'nv1': 'P1', 'nv2': 'P2'},
    'K': K, 'ddim_steps': DDIM_STEPS, 'ddim_eta': DDIM_ETA,
    'max_test_segments': MAX_TEST_SEGMENTS,
    'full_test': MAX_TEST_SEGMENTS is None,
    'normalization': MODEL_NORMALIZATION,
    'checkpoint_manifest': checkpoint_rows if RUN_INFERENCE else [],
    'dataset_audit': dataset_rows,
    'baseline_comparison_available': baseline is not None,
    'interpretation': 'Transfer/zero-shot RMN only; not an equal-training-distribution benchmark.',
    'device': str(DEVICE), 'torch_version': torch.__version__,
    'python_version': platform.python_version(),
}
(REPORT_DIR / 'run_info.json').write_text(json.dumps(run_info, indent=2), encoding='utf-8')

archive_path = OUT.parent / 'qtdb_zero_shot_rmn_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUT))
print('Saved:', archive_path)


## Cách diễn giải

- `zero_shot_summary_full_rmn.csv`: khả năng của từng model đơn nhiễu trên toàn bộ RMN, gồm cả nhiễu hỗn hợp và Clean.
- `zero_shot_summary_matching_subset.csv`: DDPM-BW chỉ trên hàng BW, DDPM-EM chỉ trên hàng EM, DDPM-MA chỉ trên hàng MA.
- `comparison_matching_subset_summary.csv`: so sánh zero-shot checkpoint với FGDAE/baseline trên đúng cùng segment, nếu đã thêm `joint_metrics_per_segment.csv`.
- `zero_shot_snr_by_rmn_composition.png`: độ tổng quát qua tám thành phần artifact RMN.

Không dùng bảng này để khẳng định model nào tốt hơn dưới cùng điều kiện train. Benchmark công bằng chính thức vẫn cần DDPM được huấn luyện trên RMN Dataset 1/2.